# 10 · Instruction Fine-Tuning Format

In plain English, this notebook is about **how you write down your training data** so that an instruction-following chatbot can learn from it. Models like ChatGPT didn't just read the internet — they were also shown thousands of **(question → ideal answer)** pairs and trained to produce the answer. That step is called **instruction fine-tuning**, or more formally **supervised fine-tuning (SFT)**.

You already know how to load models, tokenize text, and run a training loop. The missing piece is **format**: turning your raw labeled rows (like a lead with a `lead_intent` of `hot`/`warm`/`cold`) into the exact *text shape* the model expects. Get the format right and the actual fine-tuning in the next notebooks (LoRA, QLoRA) is almost copy-paste.

Everything here is plain Python and tiny data — no GPU, no model downloads required.

## What you'll learn

- What **instruction fine-tuning / supervised fine-tuning (SFT)** actually is: showing the model **(prompt → desired response)** pairs.
- **Prompt-template design**: why a *consistent* template matters, the classic **Alpaca** template (Instruction / Input / Response), a simpler **chat** template, and what a **system prompt** is for.
- How to turn our **lead dataset** into instruction examples: build a natural-language **prompt** from each lead's fields and a one-word **response** (`hot`/`warm`/`cold`).
- Two common on-disk shapes, written to a real **`instructions.jsonl`** file:
  - `{"prompt": ..., "completion": ...}`
  - chat `{"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]}`
- **Loss masking** — the intuitive idea that during SFT we score the model only on the **response** tokens, not the prompt.
- A peek at **`tokenizer.apply_chat_template`**, the standard way chat models format messages.

## Why this matters for fine-tuning

Fine-tuning is "show, don't tell." You don't explain rules to the model; you **show it examples** of the behavior you want, over and over, until it imitates them. For an instruction model, each example is a **prompt** (what the user says) paired with the **response** you wish the model had given.

The single biggest beginner mistake is treating data prep as an afterthought. In practice:

- If your examples are formatted **inconsistently**, the model gets confused about where the question ends and the answer begins.
- If you format the data one way for training but prompt the model a **different** way at test time, performance falls off a cliff.

So this notebook is the bridge between "I have labeled data" and "I can run LoRA/QLoRA." The `instructions.jsonl` file we build here is *literally* the input file the next two notebooks load. Nail the format now and the rest is easy.

## Setup

Run the cell below once. The `%pip install` line is **commented out** — uncomment it only if you're on Google Colab or a fresh environment and want to try the optional `apply_chat_template` demo near the end. Everything else uses just the Python standard library (`json`, `random`).

In [ ]:
# Uncomment the next line on Colab or a fresh environment (only needed for the
# optional apply_chat_template demo at the end):
# %pip install transformers

import json     # to write/read our data as JSON Lines (.jsonl)
import random   # to generate the toy lead dataset reproducibly

print("Ready. Using only the standard library so far.")

## 1. What is instruction fine-tuning (SFT)?

A raw "base" language model is trained to **predict the next word** on giant piles of internet text. That makes it good at *continuing* text, but not at *following instructions*. Ask a base model "Summarize this email" and it might happily write a *second* email instead — because on the internet, that's often what comes next.

**Instruction fine-tuning** fixes this. We take the base model and continue training it on a curated set of **(prompt, response)** pairs:

- **prompt** — the instruction / question / input the user provides.
- **response** — the ideal answer we want the model to produce.

Because every example comes with the *correct* response as a label, this is **supervised** learning — hence **Supervised Fine-Tuning (SFT)**. Mechanically it's still "predict the next token," but now we're nudging those predictions toward the responses *we* chose. Do this across thousands of pairs and the model learns the *habit* of answering instructions.

In [ ]:
# A "training example" for SFT is just two strings: a prompt and a response.
example = {
    "prompt":   "Classify the sentiment of this review: 'The food was amazing!'",
    "response": "positive",
}

print("PROMPT:  ", example["prompt"])
print("RESPONSE:", example["response"])

# During SFT the model reads PROMPT and is trained to produce RESPONSE.
# Stack up thousands of pairs like this and the model learns to "answer".

**What this does:**

- Shows the atom of all instruction tuning: a dict with a `prompt` and a `response`. There's no magic — an SFT dataset is *just a list of these pairs*. The art is in (a) writing good prompts/responses and (b) formatting them consistently. That's the rest of this notebook.

### ✏️ Exercise

Write **two** more `(prompt, response)` pairs of your own for a task you care about (translating a phrase, extracting a name, etc.), store them in a list `my_pairs`, and `print` it. The prompt is always the *input*; the response is the *exact text you want back*.

## 2. Prompt-template design: why a consistent template matters

A **prompt template** is a fixed *skeleton* you pour each example into. Instead of writing every prompt freehand, you decide on one shape and reuse it for **every** example — at training time **and** at inference time.

Why does consistency matter so much? The model learns to recognize the template's structure as a **signal**: "when I see this shape, the next thing I should produce is an answer." If the shape keeps changing, that signal is weak and the model is unsure where the answer should start. A consistent template is like always asking your questions in the same handwriting — it's easier to read.

Two classic templates:

1. **Alpaca-style** — separate slots for an **Instruction**, an optional **Input**, and a **Response**. Great when each example has a task description plus some data.
2. **Chat-style** — a conversation made of **roles** (`system`, `user`, `assistant`). This matches how modern chat models (Llama, Mistral, Qwen, GPT) are actually trained.

We'll build both below.

In [ ]:
# ---- The classic Alpaca-style template ----
# {instruction} = what to do, {input} = the data to do it on, then "### Response:"
ALPACA_TEMPLATE = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
"""

# Fill the slots for one example:
prompt_text = ALPACA_TEMPLATE.format(
    instruction="Classify the lead's intent as hot, warm, or cold.",
    input="family size 4, income 100000, owns home, requested a quote in May, condition urgent",
)
response_text = "hot"

print(prompt_text + response_text)

**What this does:**

- `ALPACA_TEMPLATE` is one fixed string with `{instruction}` and `{input}` placeholders.
- `.format(...)` fills the slots. The template ends with `### Response:` and a newline, so the model knows *exactly* where its answer should begin.
- The **response** (`"hot"`) is appended right after. During training the model sees prompt **+** response together; at inference we give it everything **up to** `### Response:` and let it generate the rest.

> The leading paragraph ("Below is an instruction...") is called a **preamble**. It's optional but helps a base model understand the format. Keep it identical across all examples.

In [ ]:
# ---- A simpler chat-style template ----
# A short conversation: an optional system message, the user's turn, then the
# assistant's turn (the part we want the model to learn to produce).
SYSTEM_PROMPT = "You are a sales assistant. Classify each lead's intent as hot, warm, or cold."

def render_chat(user_text, assistant_text=""):
    """Render a tiny system/user/assistant conversation as one string."""
    return (
        f"<|system|>\n{SYSTEM_PROMPT}\n"
        f"<|user|>\n{user_text}\n"
        f"<|assistant|>\n{assistant_text}"
    )

print(render_chat(
    user_text="Lead: family size 1, income 25000, rents home, newsletter signup in Feb, just browsing.",
    assistant_text="cold",
))

**What this does:**

- A **system prompt** sets the model's *role and rules* once, up front ("You are a sales assistant..."). It's the standing instruction that applies to the whole conversation.
- `render_chat` wraps each turn with simple role markers (`<|system|>`, `<|user|>`, `<|assistant|>`). Real chat models use their own special tokens, but the *idea* — label each turn with a role — is exactly this.
- The `assistant_text` is what we want the model to generate. At inference we'd call `render_chat(user_text)` with an **empty** assistant slot and let the model fill it in.

> System prompt vs. instruction: the **system prompt** is the persistent role ("you are X, always do Y"); the **user** message is the specific request ("here is *this* lead"). Putting shared rules in the system prompt keeps every user turn short.

### ✏️ Exercise

Change `SYSTEM_PROMPT` to a different personality (e.g. *"You are a blunt analyst who only ever replies with a single lowercase word."*), then re-run `render_chat("Lead: ...", "warm")`. Notice the **structure** is unchanged — only the system text moved. That separation is the whole point of a system prompt.

## 3. Our raw data: the lead dataset

We'll reuse the same **synthetic lead dataset** that runs through this whole course. Each lead is a dict of fields (family size, income, etc.) plus a computed **`lead_intent`** label that is one of `"hot"`, `"warm"`, or `"cold"`.

Run the generator below. It's deterministic (`random.seed(42)`), so everyone gets the same 200 leads.

In [ ]:
import random
random.seed(42)
MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
CTAS = ["requested_quote","booked_demo","downloaded_brochure","newsletter_signup"]
CONDITIONS = ["urgent","exploring","just_browsing"]

def make_lead():
    family_size = random.randint(1, 6)
    income = random.choice([25000,40000,55000,70000,85000,100000,120000,150000])
    rent_or_own = random.choice(["rent","own"])
    cta = random.choice(CTAS)
    engagement_month = random.choice(MONTHS)
    current_condition = random.choice(CONDITIONS)
    score = 0
    if cta in ("requested_quote","booked_demo"): score += 2
    elif cta == "downloaded_brochure": score += 1
    if rent_or_own == "own": score += 1
    if income >= 80000: score += 1
    if family_size >= 4: score += 1
    if current_condition == "urgent": score += 2
    elif current_condition == "exploring": score += 1
    lead_intent = "hot" if score >= 5 else ("warm" if score >= 3 else "cold")
    return {"family_size": family_size, "income": income, "rent_or_own": rent_or_own,
            "cta": cta, "engagement_month": engagement_month,
            "current_condition": current_condition, "lead_intent": lead_intent}

leads = [make_lead() for _ in range(200)]

print("Total leads:", len(leads))
print("First lead:", leads[0])

# Quick label balance check:
from collections import Counter
print("Label counts:", Counter(l["lead_intent"] for l in leads))

**What this does:**

- Builds **200** lead dicts. Each has six input fields plus the target `lead_intent`.
- The `score` logic is just our *pretend* business rule for labeling. In a real project these labels would come from your CRM or human annotators — but a clean rule lets us focus on **format**, not on guessing.
- `Counter(...)` shows the class balance. You'll typically see more `warm`/`cold` than `hot`; that's realistic and fine for learning.

### ✏️ Exercise

Print the **last** lead (`leads[-1]`) and, by reading the `make_lead` scoring rules, try to predict its `lead_intent` **before** you look at the field. Then check whether you were right. This builds intuition for what the model will have to learn.

In [ ]:
# Your turn:
# print(leads[-1])

## 4. Turn each lead into an instruction example

Now the key step: convert a structured lead dict into a **natural-language prompt** and a **one-word response**. We'll describe the lead's fields in a sentence and ask the model for the intent. The response is just the `lead_intent` label.

We'll use a compact, consistent template (a refined version of the one suggested for this course):

```
You are a sales assistant. Classify the lead's intent as hot, warm, or cold.
Lead: family size {family_size}, income {income}, {rent_or_own}s home, engaged via {cta} in {engagement_month}, current condition {current_condition}.
Intent:
```

and the response is a single space + the label, e.g. `" hot"`.

In [ ]:
PROMPT_TEMPLATE = (
    "You are a sales assistant. Classify the lead's intent as hot, warm, or cold.\n"
    "Lead: family size {family_size}, income {income}, {rent_or_own}s home, "
    "engaged via {cta} in {engagement_month}, current condition {current_condition}.\n"
    "Intent:"
)

def lead_to_example(lead):
    """Return a (prompt, completion) dict for one lead."""
    prompt = PROMPT_TEMPLATE.format(
        family_size=lead["family_size"],
        income=lead["income"],
        rent_or_own=lead["rent_or_own"],
        cta=lead["cta"],
        engagement_month=lead["engagement_month"],
        current_condition=lead["current_condition"],
    )
    completion = " " + lead["lead_intent"]   # leading space: the model continues "Intent:"
    return {"prompt": prompt, "completion": completion}

# Build an instruction example for every lead:
examples = [lead_to_example(l) for l in leads]
print("Built", len(examples), "instruction examples.")

**What this does:**

- `PROMPT_TEMPLATE` is our fixed skeleton. Every lead is poured into the **same** shape.
- `lead_to_example` fills the slots from one lead's fields and sets `completion` to a **leading-space + label** (e.g. `" hot"`). The leading space matters: the prompt ends with `Intent:` and most tokenizers expect the next word to start with a space. This little detail keeps tokenization clean.
- We now have a list of 200 `{"prompt", "completion"}` dicts — a real SFT dataset.

In [ ]:
# Let's actually LOOK at 3 fully rendered examples (prompt + completion):
for ex in examples[:3]:
    print("=" * 60)
    print(ex["prompt"], end="")     # prompt ends with "Intent:" (no newline)
    print(ex["completion"])         # the model is trained to produce this
print("=" * 60)

# Expected: three blocks, each ending with " hot", " warm", or " cold".

**What this does:**

- Prints the **prompt immediately followed by the completion**, exactly as the model will see them concatenated during training.
- Read one carefully: the description sentence, then `Intent:`, then the label. A human can answer it — which is a good sanity check that the model can learn it too.

> Rule of thumb: **if you can't answer your own prompt from the text alone, neither can the model.** Make sure every fact the answer depends on is actually present in the prompt.

### ✏️ Exercise

Add a new field to the prompt. For example, append `" Engagement recency: recent."` to the template and re-run the rendering for `examples[:1]` (rebuild it after editing the template). The point: changing the template is easy, but you must rebuild **all** examples so they stay consistent. Mismatched templates are the #1 bug in SFT data.

In [ ]:
# Your turn (copy PROMPT_TEMPLATE, tweak it, rebuild, and print one example):
# NEW_TEMPLATE = PROMPT_TEMPLATE.replace("Intent:", "Note: high priority.\nIntent:")
# print(NEW_TEMPLATE.format(**{k: leads[0][k] for k in
#       ["family_size","income","rent_or_own","cta","engagement_month","current_condition"]}))

## 5. Save the data as JSONL — shape (a): `prompt` / `completion`

**JSONL** ("JSON Lines") is the standard format for fine-tuning datasets: **one JSON object per line**. It's easy to stream, easy to append to, and supported by virtually every training library (Hugging Face `datasets`, OpenAI, Axolotl, etc.).

Shape (a) is the simplest: each line is `{"prompt": ..., "completion": ...}`.

In [ ]:
# Write the first 5 examples to a JSONL file (small, so we can read it back).
with open("instructions.jsonl", "w", encoding="utf-8") as f:
    for ex in examples[:5]:
        line = json.dumps(ex)          # turn the dict into a JSON string
        f.write(line + "\n")           # one object per line

print("Wrote 5 lines to instructions.jsonl")

# Read it back and show the raw lines:
with open("instructions.jsonl", encoding="utf-8") as f:
    for i, raw_line in enumerate(f):
        print(f"line {i}: {raw_line.rstrip()}")

**What this does:**

- `json.dumps(ex)` converts a Python dict into a **JSON string**; we write it followed by `"\n"` so each example sits on its **own line**.
- Reading it back, each line is a complete, self-contained JSON object. That's the whole idea of JSONL: line-by-line, append-friendly, no giant array to load all at once.
- `\n` inside a prompt is escaped as `\\n` in the JSON — that's correct and gets un-escaped when you load it.

### ✏️ Exercise

Load the file back with `json.loads` (one `json.loads(line)` per line into a list) and confirm the round-trip is lossless: check that `loaded[0] == examples[0]` prints `True`.

## 6. Save the data as JSONL — shape (b): chat `messages`

Modern chat models expect a **list of role-tagged messages**. The on-disk shape is:

```json
{"messages": [
   {"role": "system",    "content": "..."},
   {"role": "user",      "content": "..."},
   {"role": "assistant", "content": "..."}
]}
```

This is the format most current SFT recipes (and providers like OpenAI) prefer, because the tokenizer can apply the model's *own* chat formatting automatically (more on that in Section 8).

In [ ]:
SYSTEM_MSG = "You are a sales assistant. Classify each lead's intent as hot, warm, or cold."

def lead_to_chat(lead):
    """Return a chat-format example: system + user + assistant messages."""
    user_text = (
        "Lead: family size {family_size}, income {income}, {rent_or_own}s home, "
        "engaged via {cta} in {engagement_month}, current condition {current_condition}. "
        "What is the intent?"
    ).format(**{k: lead[k] for k in
                ["family_size","income","rent_or_own","cta","engagement_month","current_condition"]})
    return {"messages": [
        {"role": "system",    "content": SYSTEM_MSG},
        {"role": "user",      "content": user_text},
        {"role": "assistant", "content": lead["lead_intent"]},
    ]}

chat_examples = [lead_to_chat(l) for l in leads]

# Pretty-print one chat example so we can see the structure:
print(json.dumps(chat_examples[0], indent=2))

**What this does:**

- `lead_to_chat` builds a 3-message conversation: the **system** rule, the **user** request, and the **assistant** answer (the label).
- `json.dumps(..., indent=2)` pretty-prints it so the nested structure is readable.
- Notice the **same lead** can be expressed in *either* shape — `prompt`/`completion` (Section 5) or chat `messages`. They carry the same information; you pick the shape your training tool expects.

In [ ]:
# Write the chat-format examples to their own JSONL file (first 5):
with open("instructions_chat.jsonl", "w", encoding="utf-8") as f:
    for ex in chat_examples[:5]:
        f.write(json.dumps(ex) + "\n")

print("Wrote 5 chat-format lines to instructions_chat.jsonl")

# Show the first raw line (compact, one object per line):
with open("instructions_chat.jsonl", encoding="utf-8") as f:
    print(f.readline().rstrip())

**What this does:**

- Saves the chat shape to a second file. Same JSONL rules: **one object per line**, compact (no `indent`) so each example is exactly one line. You now have **both** shapes on disk.

### ✏️ Exercise

Append an extra `user` turn (e.g. *"Just give one word."*) and another `assistant` turn to `chat_examples[0]["messages"]`, then `print(json.dumps(..., indent=2))`. Multi-turn conversations are just **longer message lists**; the format doesn't change.

## 7. Loss masking: the model should learn to *answer*, not to *repeat the question*

Here's a subtle but crucial idea. During training, the model reads the **whole** sequence (prompt + response) and, at every position, predicts the next token. By default the training **loss** (the "how wrong was I?" score) would be computed over **all** of those tokens — including the prompt.

But we don't want the model to get good at **reciting the prompt** — the user *gives* us the prompt; the model never has to produce it. We only care that it produces a good **response**. So in SFT we usually apply **loss masking**: we compute the loss **only on the response tokens** and **ignore** (mask out) the prompt tokens.

Intuitively:

- Prompt tokens → *given to the model, not scored.* ("You already know this; don't waste learning on it.")
- Response tokens → *scored.* ("This is what you must learn to generate.")

This makes training focus its effort exactly where it matters. Let's make it concrete with a toy label mask — no real tokenizer needed.

In [ ]:
# Toy illustration of loss masking, using whitespace "tokens" for clarity.
prompt   = "Intent:"
response = " hot"

prompt_tokens   = prompt.split()        # ['Intent:']
response_tokens = response.split()      # ['hot']
all_tokens = prompt_tokens + response_tokens

# Convention: -100 means "ignore this position when computing loss".
# Prompt positions get -100; response positions keep a real label (here, the
# token itself stands in for its id).
IGNORE = -100
labels = [IGNORE] * len(prompt_tokens) + response_tokens

for tok, lab in zip(all_tokens, labels):
    role = "PROMPT  (masked, not scored)" if lab == IGNORE else "RESPONSE (scored)"
    print(f"{tok!r:12} -> label={lab!r:8} | {role}")

**What this does:**

- We split the prompt and response into toy "tokens" so the idea is visible without a real tokenizer.
- We build a `labels` list where **prompt positions are `-100`** and **response positions keep their real label**.
- The value **`-100`** is the standard "ignore index" in PyTorch / Hugging Face loss functions: any position labeled `-100` is **skipped** when computing the loss. That's literally how loss masking is implemented under the hood.

> The good news: high-level SFT tools (like Hugging Face's `SFTTrainer` and the LoRA recipes in the next notebooks) can do this masking **for you** when you give them the prompt/response or chat format. You mostly just need to *understand* it so the results make sense — and so you know why a clean prompt/response split matters.

### ✏️ Exercise

Change `response` to two words (e.g. `" very hot"`) and re-run the masking cell above. Confirm that **only the response positions** get real labels while the `Intent:` prompt position still shows `-100`. This is the core mechanic of "learn to answer, not to echo."

## 8. The real thing: `tokenizer.apply_chat_template`

When you use a **chat model**, you should *not* invent your own `<|user|>` markers — each model family has its **own** special tokens and formatting. The tokenizer knows them, and exposes a method that turns a `messages` list into the exact string (or token ids) that model was trained on:

```python
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
```

- `messages` is the same `[{"role": ..., "content": ...}]` list from Section 6.
- `add_generation_prompt=True` appends the marker that says "your turn, assistant" — use it at **inference** time so the model starts answering.
- It needs a **chat model's** tokenizer (one that ships a chat template), so the cell below is **commented out** and optional — uncomment it if you have `transformers` installed and don't mind a small download.

In [ ]:
# OPTIONAL — needs `transformers` and a small model download.
# Uncomment to see your messages rendered in a real model's chat format.
#
# from transformers import AutoTokenizer
# tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
# messages = chat_examples[0]["messages"]
# rendered = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
# print(rendered)
#
# # You'll see the model's OWN special tokens wrapping each role, e.g.
# # <|im_start|>system ... <|im_end|> <|im_start|>user ... <|im_end|> ...
# # That's why you let the tokenizer do it instead of hand-writing markers.

**What this does:**

- Demonstrates the **standard, correct** way to format chat data: hand the tokenizer your `messages` and let it apply the model's official template. This is why **shape (b)** is so convenient — `apply_chat_template` consumes it directly, so you never memorize each model's special tokens.

> Takeaway: write your data as plain `messages`, and let `apply_chat_template` (inside the trainer) turn it into model-specific text. Portable data, model-specific formatting — best of both worlds.

## 9. Two ways to frame the *same* task

We just framed lead-scoring as a **generative / instruction** task: prompt the model and have it **write** the word `hot`/`warm`/`cold`. That's flexible — the model can also explain itself, output multiple labels, etc.

There's a second, simpler framing you saw earlier in the course: **classification**. There, the model has a small **classification head** that outputs a probability for each of the 3 classes, and you pick the highest. No prompt template, no text generation — just `input → one of N labels`.

| | Instruction / generative (this notebook) | Classifier framing |
|---|---|---|
| Output | **Text** ("hot") the model generates | A **label index** (0/1/2) from a head |
| Data shape | prompt + response (or chat messages) | text + integer label |
| Model class | `AutoModelForCausalLM` | `AutoModelForSequenceClassification` |
| Loss | next-token loss on the **response** (masked) | cross-entropy over the 3 classes |
| Strength | flexible, can explain, handles open-ended tasks | simpler, often more accurate for fixed labels |

Neither is "better" — they're different tools. **Notebook 11 uses LoRA** to fine-tune efficiently, and the **capstone** shows *both* framings on this exact lead task so you can compare them head-to-head.

In [ ]:
# Same lead, two framings, side by side:
lead = leads[0]

# (A) Instruction / generative target: a STRING the model must produce.
gen_prompt   = lead_to_example(lead)["prompt"]
gen_response = lead_to_example(lead)["completion"]   # e.g. " warm"

# (B) Classifier target: an INTEGER label index.
LABELS = ["cold", "warm", "hot"]
clf_label_id = LABELS.index(lead["lead_intent"])      # e.g. 1 for "warm"

print("Generative response :", repr(gen_response))
print("Classifier label id :", clf_label_id, "->", LABELS[clf_label_id])

**What this does:**

- Takes one lead and produces **both** kinds of target: a **string** (`" warm"`) for the generative framing and an **integer** (`1`) for the classifier framing.
- This is the bridge between this notebook and the rest of the course: the *data* is the same; only the *format of the answer* changes depending on which model class you fine-tune.

### ✏️ Exercise

For `leads[5]`, print its generative response string (`lead_to_example(leads[5])["completion"]`) **and** its classifier label id (`LABELS.index(...)`). Then explain to yourself in one sentence why the generative version needs the prompt template but the classifier version does not.

## Common mistakes & how to debug them

- **Train/inference template mismatch.** If you train with `"Intent:"` but at test time prompt with `"Lead intent ->"`, accuracy collapses. **Use the identical template both times.** Save your template alongside your data.
- **Forgetting the leading space on the completion.** Tokenizers usually treat `" hot"` and `"hot"` as *different* tokens. If your prompt ends with `Intent:` (no trailing space), put the space on the completion (`" hot"`), and be consistent everywhere.
- **Inconsistent label text.** `"Hot"`, `"hot"`, `"HOT"`, and `"hot."` are four different targets to the model. Pick one exact spelling/casing and stick to it. `Counter` your labels to spot stragglers.
- **Putting the answer in the prompt.** If your prompt accidentally already contains the label, the model learns to copy it and fails on real data. Re-read a few rendered examples and make sure the answer is **only** in the response.
- **Pretty-printing your JSONL.** JSONL must be **one object per line**. Don't use `json.dump(..., indent=2)` when writing the file — that spreads one object across many lines and breaks line-based loaders. (Pretty-print only for *viewing*, as in Section 6.)
- **No loss masking → model echoes the question.** If your fine-tuned model repeats the prompt back instead of answering, you (or your trainer) likely scored the prompt tokens. Make sure only the **response** is scored.
- **Missing fields in the template.** A `KeyError` from `.format(...)` means a placeholder like `{income}` has no matching key. Double-check the field names match your data dict exactly.

## Summary

- **Instruction fine-tuning (SFT)** = training a model on **(prompt → desired response)** pairs so it learns to *answer*, not just continue text.
- A **consistent prompt template** is essential. We used the **Alpaca** template (Instruction / Input / Response) and a **chat** template (system / user / assistant), and saw what a **system prompt** is for.
- We turned the **lead dataset** into instruction examples: a natural-language prompt built from each lead's fields, with a one-word completion (`hot`/`warm`/`cold`), and rendered several in full.
- We saved the data as **JSONL** in two shapes: `{"prompt", "completion"}` (`instructions.jsonl`) and chat `{"messages": [...]}` (`instructions_chat.jsonl`) — **one JSON object per line**.
- **Loss masking**: during SFT we score **only the response tokens** (prompt positions get the ignore-index `-100`) so the model learns to answer, not echo.
- **`tokenizer.apply_chat_template`** is the standard way to render chat `messages` into a model's *own* format — write portable `messages`, let the tokenizer handle the special tokens.
- The **same task** can be framed as **generative** (this notebook) or as a **classifier**; the data is the same, only the answer format differs.

The `instructions.jsonl` you built here is exactly the file the next notebooks load. Format mastered — time to actually train.

## What to learn next

Next up: **`11_lora_finetuning.ipynb`**. You'll take the instruction data from this notebook and run a **real, parameter-efficient fine-tune** using **LoRA** — a technique that trains a tiny number of extra weights instead of the whole model, so it fits on a laptop or a free Colab GPU. You already have the formatted data and you understand loss masking; LoRA just makes the *training step* cheap. See you there.